# 03 · Evaluate extraction quality with MLflow

Score the extraction against the **hand-labelled golden set** with `mlflow.genai.evaluate`,
driven **entirely from tables** — each row is `{inputs, outputs, expectations}` read from
`extracted_flat` (notebook 02) + `golden_labels` (notebook 00). **No live call**, no
`predict_fn`, no `spark.sql` in a scorer — so the same scorers apply whether extraction ran
as this batch pipeline or behind a model-serving endpoint.

- **Code scorers** — tolerant, null-aware match (substring + date normalization); overall,
  per field, and coverage recall.
- **LLM judge** — a custom judge via the deploy client for the semantic insurer-name match
  (and it correctly fails template placeholders like "Insert Insurer name").

In [0]:
%pip install --quiet --upgrade "mlflow[databricks]>=3.1"
%restart_python

In [0]:
CATALOG, SCHEMA = "users", "scott_mckean"
JUDGE_ENDPOINT = "databricks-claude-sonnet-4"
import mlflow, mlflow.deployments, re
from mlflow.genai.scorers import scorer
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/document_intelligence_eval")
deploy_client = mlflow.deployments.get_deploy_client("databricks")
print("MLflow", mlflow.__version__, "| experiment set for", user)

## 1 · Build the eval dataset from the tables

`outputs` from `extracted_flat`, `expectations` from `golden_labels`. Null ground truth is
encoded as `""` (a valid expectation value); the scorers treat empty as *expected null*.

In [0]:
SCALAR = ["document_type", "insurer_name", "insured_name", "policy_number", "effective_date", "expiration_date"]
DATES = {"effective_date", "expiration_date"}

gold = {r["filename"]: r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.golden_labels").collect()}
ext = {r["filename"]: r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.extracted_flat").collect()}
parsed = {r["filename"]: r["full_text"] for r in spark.table(f"{CATALOG}.{SCHEMA}.parsed_text").collect()}

eval_data = []
for fn, g in gold.items():
    if fn not in ext:
        continue
    eval_data.append({
        "inputs": {
            "filename": fn,
            "full_text": parsed.get(fn, ""),
        },
        "outputs": {f: (ext[fn].get(f) or "") for f in SCALAR + ["coverage_types"]},
        "expectations": {
            **{f: (g[f] if g[f] is not None else "") for f in SCALAR},
            "coverage_types": list(g["coverage_types"]) if g["coverage_types"] is not None else [],
        },
    })
print(f"{len(eval_data)} labelled docs in eval dataset")

## 2 · Tolerant, null-aware scorers + LLM judge

Code scorers use **substring + date-normalized** matching — tolerant of formatting noise
but deterministic and cheap. The LLM judge calls a foundation model for **semantic
insurer-name matching** (and correctly rejects template placeholders like "Insert Insurer
name").

In [0]:
def _norm(v):
    if v is None: return ''
    s = ' '.join(str(v).strip().lower().split())
    return re.sub('[^a-z0-9 ]', '', s).strip()

def _ndate(v):
    if not v: return ''
    m = re.findall(r'\d+', str(v))
    if len(m) >= 3:
        mo, d, y = m[0], m[1], m[2]
        if len(y) == 2: y = ("19" if int(y) >= 50 else "20") + y
        return f"{int(mo):02d}/{int(d):02d}/{y}"
    return _norm(v)

def _score(field, got, exp):
    nullx = exp is None or (isinstance(exp, str) and exp.strip() == "")
    if field in DATES:
        g = _ndate(got)
        return (1.0 if g == "" else 0.0) if nullx else (1.0 if g == _ndate(exp) else 0.0)
    g, e = _norm(got), _norm(exp)
    if nullx: return 1.0 if g == "" else 0.0
    if g == "": return 0.0
    return 1.0 if (g == e or e in g or g in e) else 0.0

@scorer
def field_accuracy(outputs, expectations) -> float:
    """Mean tolerant match across the scalar fields."""
    s = [_score(f, (outputs or {}).get(f), expectations.get(f)) for f in SCALAR]
    return sum(s) / len(s)

def _field_scorer(field):
    @scorer(name=f"acc_{field}")
    def _s(outputs, expectations) -> float:
        return _score(field, (outputs or {}).get(field), expectations.get(field))
    return _s
per_field_scorers = [_field_scorer(f) for f in SCALAR]

@scorer(name="coverage_recall")
def coverage_recall(outputs, expectations) -> float:
    """Fraction of expected coverage types found in the extracted text."""
    exp = expectations.get("coverage_types") or []
    if not exp: return 1.0
    text = _norm((outputs or {}).get("coverage_types"))
    found = 0
    for c in exp:
        nc = _norm(c); w = nc.split()[0] if nc else ''
        if nc and (nc in text or (len(w) > 4 and w in text)): found += 1
    return found / len(exp)

@scorer(name="insurer_name_llm_judge")
def insurer_llm_judge(outputs, expectations) -> float:
    """LLM judge: same insurer? Placeholders are NOT a match."""
    exp = expectations.get("insurer_name"); got = (outputs or {}).get("insurer_name")
    if not exp or not str(exp).strip():
        return 1.0 if not _norm(got) else 0.0
    prompt = (
        f'Expected insurer: "{exp}"\nExtracted insurer: "{got}"\n'
        "Do these refer to the same insurance company (ignore punctuation/suffixes; a "
        "placeholder like \"Insert Insurer name\" or \"Carrier A\" is NOT a match)? "
        "Answer only YES or NO."
    )
    resp = deploy_client.predict(endpoint=JUDGE_ENDPOINT,
        inputs={"messages": [{"role": "user", "content": prompt}], "max_tokens": 3, "temperature": 0})
    return 1.0 if resp["choices"][0]["message"]["content"].strip().lower().startswith("y") else 0.0

## 3 · Run `mlflow.genai.evaluate` (table-driven)

No `predict_fn` needed — we pass pre-computed `{inputs, outputs, expectations}` directly.
Each row produces a **trace with full document text, extracted fields, and ground truth**
visible in the MLflow experiment UI.

In [0]:
results = mlflow.genai.evaluate(
    data=eval_data,
    scorers=[field_accuracy, coverage_recall, insurer_llm_judge] + per_field_scorers,
)
print("\n=== Aggregate metrics ===")
for k, v in sorted(results.metrics.items()):
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

In [0]:
detail = mlflow.search_traces(run_id=results.run_id)
print("Logged", len(detail), "traces. Open the run in the MLflow UI for per-field scores.")
display(detail)

## What you get

Real numbers on messy public docs (a recent run): `field_accuracy ~0.78`, dates &
`policy_number ~0.89`, `insurer_name ~0.56` by code match but only **~0.44 by the LLM
judge** — because the judge correctly rejects template placeholders the string match lets
through. That gap is the point: code scorers are cheap and deterministic, LLM judges catch
semantic/placeholder errors.

- A **labelled benchmark run** + **a trace per document** in the experiment.
- Fully **table-driven** — repoint at a batch table or a serving endpoint's logged output
  and re-run for **regression testing** after prompt / field / model changes.

**Extend it:** add the customer's declarations/SOV docs with property-underwriting labels
(flood zone, construction, year built, prior claims, hazmat, sprinklers), add LLM judges
for other semantic fields, or align a judge to an underwriter's labels with **MemAlign**.